In [ ]:
# NOTEBOOK PARAMETERS
selected_model = 'neptune_1'
ensemble_size = 8
years = [2019,2020,2021]

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import xarray as xr
import numpy as np
import os
import sys

sys.path.append('library')
from atmocean.visualize import (
    get_colormap_from_cmap,
    get_sst_bias_colormap,
    plot_dmi_timeseries,
    plot_map_on_grid,
    discrete_cmap,
)
from atmocean.enso import (
    monthly_resample_with_prediction_timedelta,
    monthly_groupby_with_prediction_timedelta,
    compute_eke,
    compute_tke,
)
from atmocean.utils import (
    clean_forecasts,
    remove_ocean_variables_suffix,
    align_prediction_timedelta_optim,
)
os.system('hostname')
%load_ext autoreload
%autoreload 2

In [ ]:
# get the forecasting horizon for this run
start_date = '01-01'
end_date = '12-31'
years = [2019,2020,2021]
lead_times = [0, 10, 20, 30, 40, 50, 60]

main_dir = 'PATH-TO-MAIN-DIRECTORY'

data_dir = os.path.join(main_dir, 'data')
forecast_dir = os.path.join(data_dir, 'forecasts')

climatology_src = os.path.join(forecast_dir, f'climatology-from_1993_to_2017.zarr')

if ensemble_size is None:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}/physical_coherency/eke')
else:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}-ENS-{ensemble_size:02d}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}-ENS-{ensemble_size:02d}/physical_coherency/eke')

os.makedirs(out_img_dir, exist_ok=True)

plot_kwargs = dict(dpi=100, bbox_inches='tight')
save_figures = True

In [ ]:
from dask.distributed import Client, LocalCluster, progress, wait
from tqdm import tqdm
import numpy as np
import dask
import ctypes

# spill to disk before hitting hard memory limits (avoids KilledWorker)
dask.config.set({
    'distributed.worker.memory.target': 0.6,
    'distributed.worker.memory.spill': 0.7,
    'distributed.worker.memory.pause': 0.8,
    'distributed.worker.memory.terminate': 0.95,
})

cpu_count = os.cpu_count() or 16

n_workers = min(16, cpu_count // 2)
threads_per_worker = 2  # o 4

memory_limit = 948 // n_workers

print(f'memory_limit {memory_limit}GB')
print(f'n_workers {n_workers}')
print(f'threads_per_worker {threads_per_worker}')
cluster = LocalCluster(
    n_workers=n_workers, threads_per_worker=threads_per_worker,
    memory_limit=f'{memory_limit}GB',
    local_directory=os.path.join(main_dir, 'tmp_dask'),  # spill directory
)
client = Client(cluster)

def _trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client.run(_trim_memory)
client

In [ ]:
# open forecasts
target_mib = 128
dask.config.set({'array.chunk-size': f'{target_mib}MiB'})

chunks = {'time': 5, 'prediction_timedelta': 1, 'latitude': -1, 'longitude': -1}
if ensemble_size is not None:
    chunks['ensemble_member'] = 1

forecasts = xr.open_mfdataset(
    forecasts_srcs, chunks=chunks, parallel=True, engine='zarr', preprocess=clean_forecasts
).astype(np.float32) # select only meridional and zonal velocities
# manipulate the time dimension to align
forecasts = forecasts.sel(prediction_timedelta=pd.to_timedelta(lead_times, unit='days')).isel(depth=0, level=0, missing_dims='ignore')
# remove ocean variables suffix (oras, cglo, glor)
forecasts = remove_ocean_variables_suffix(forecasts)[[f'uo','vo']]
if ensemble_size is not None:
    forecasts = forecasts.isel(ensemble_member=0).drop_vars('ensemble_member')

In [ ]:
forecasts_eke = compute_eke(forecasts)
forecasts_eke = client.persist(forecasts_eke)
progress(forecasts_eke)
client.run(_trim_memory)

In [ ]:
forecasts_tke = compute_tke(forecasts)
forecasts_tke = client.persist(forecasts_tke)
progress(forecasts_tke)
client.run(_trim_memory)

In [ ]:
forecasts_eke = align_prediction_timedelta_optim(forecasts_eke)
forecasts_tke = align_prediction_timedelta_optim(forecasts_tke)

In [ ]:
forecasts_eke, forecasts_tke = client.persist([forecasts_eke, forecasts_tke])
progress([forecasts_eke, forecasts_tke])
wait([forecasts_eke, forecasts_tke])

In [ ]:
from matplotlib.ticker import ScalarFormatter
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
from matplotlib import pyplot as plt
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.crs as ccrs

In [ ]:
grid_list = [
    'gs[0,0:2]',
    'gs[0,2:4]',
    'gs[0,4:6]',
    'gs[1,0:2]',
    'gs[1,2:4]',
    'gs[1,4:6]',
    'gs[2,2:4]',
]

In [ ]:
forecasts_mean_eke = forecasts_eke.mean(dim='time').load()
forecasts_mean_tke = forecasts_tke.mean(dim='time').load()

In [ ]:
cmap, norm, levels = get_colormap_from_cmap('Spectral_r', 0.0, 0.1) # get_sst_bias_colormap(0, 0.1)

eke_list = []
for i,lt in enumerate(lead_times):
    eke_list.append(
        forecasts_mean_eke.isel(prediction_timedelta=i)
    )

central_longitude = 180
lats, lons = eke_list[0].latitude.data, eke_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(10,5.5))
fig.suptitle(f'Eddy Kinetic Energy', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(3, 6, figure=fig)
axs = []
eps = 0.0 # 1e-2
for i,lt in enumerate(lead_times):
    im, ax = plot_map_on_grid(
        bias=eke_list[i], grid=eval(grid_list[i]), title='Reanalysis' if i == 0 else f'Forecast t+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap, norm=norm, levels=levels
    )
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='m²/s²', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=-0.3, hspace=0.5
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'eddy_kinetic_energy_spatial.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
cmap, norm, levels = get_colormap_from_cmap('Spectral_r', 0.0, 0.1) # get_sst_bias_colormap(0, 0.1)

tke_list = []
for i,lt in enumerate(lead_times):
    tke_list.append(
        forecasts_mean_tke.isel(prediction_timedelta=i)
    )

central_longitude = 180
lats, lons = tke_list[0].latitude.data, tke_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(10,5.5))
fig.suptitle(f'Total Kinetic Energy', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(3, 6, figure=fig)
axs = []
eps = 0.0 # 1e-2
for i,lt in enumerate(lead_times):
    im, ax = plot_map_on_grid(
        bias=tke_list[i], grid=eval(grid_list[i]), title='Reanalysis' if i == 0 else f'Forecast t+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap, norm=norm, levels=levels
    )
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='m²/s²', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=-0.3, hspace=0.5
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'total_kinetic_energy_spatial.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
cmap, norm, levels = get_colormap_from_cmap('RdBu_r', -0.05, 0.05) # get_sst_bias_colormap(-0.05, 0.05)

eke_bias_list = []
for i,lt in enumerate(lead_times[1:]):
    eke_bias_list.append(
        forecasts_mean_eke.isel(prediction_timedelta=i+1) - forecasts_mean_eke.isel(prediction_timedelta=0)
    )

central_longitude = 180
lats, lons = eke_bias_list[0].latitude.data, eke_bias_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(18,5.5))
fig.suptitle(f'Eddy Kinetic Energy Bias', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 6, figure=fig)
axs = []
# eps = 1e-3
eps = 0.0
for i,lt in enumerate(lead_times[1:]):
    im, ax = plot_map_on_grid(
        bias=eke_bias_list[i], grid=eval(grid_list[i]), title=f'Forecast t+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap, norm=norm, levels=levels
    )
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='m²/s²', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=-0.3, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'eddy_kinetic_energy_bias_spatial.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
cmap, norm, levels = get_colormap_from_cmap('RdBu_r', -0.05, 0.05) # get_sst_bias_colormap(-0.05, 0.05)

tke_bias_list = []
for i,lt in enumerate(lead_times[1:]):
    tke_bias_list.append(
        forecasts_mean_tke.isel(prediction_timedelta=i+1) - forecasts_mean_tke.isel(prediction_timedelta=0)
    )

central_longitude = 180
lats, lons = tke_bias_list[0].latitude.data, tke_bias_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(18,5.5))
fig.suptitle(f'Total Kinetic Energy Bias', fontsize=16, y=1.05)

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 6, figure=fig)
axs = []
# eps = 1e-3
eps = 0.0
for i,lt in enumerate(lead_times[1:]):
    im, ax = plot_map_on_grid(
        bias=tke_bias_list[i], grid=eval(grid_list[i]), title=f'Forecast t+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap, norm=norm, levels=levels
    )
    axs.append(ax)

# single colorbar
cax = fig.add_axes([0.25, -0.05, 0.5, 0.03]) # [left, bottom, width, height]
cbar = fig.colorbar(
    im, ax=axs, cax=cax, 
    orientation='horizontal', pad=0.05, 
    aspect=60, shrink=0.9, label='m²/s²', 
    format=ScalarFormatter(), 
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=-0.3, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'total_kinetic_energy_bias_spatial.png')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
lead_times_figure = [0, 10, 30, 60]
grid_list_top = [
    'gs[0,0:2]',
    'gs[0,2:4]',
    'gs[0,4:6]',
    'gs[0,6:8]',
]
grid_list_bottom = [
    'gs[1,0:2]',
    'gs[1,2:4]',
    'gs[1,4:6]',
    'gs[1,6:8]',
]

cmap_top, norm_top, levels_top = get_colormap_from_cmap('Spectral_r', 0.0, 0.1)
cmap_bottom, norm_bottom, levels_bottom = get_colormap_from_cmap('RdBu_r', -0.05, 0.05)

eke_list = []
eke_bias_list = []
for i,lt in enumerate(lead_times_figure):
    eke_list.append(
        forecasts_mean_eke.isel(prediction_timedelta=lt//10)
    )
    eke_bias_list.append(
        forecasts_mean_eke.isel(prediction_timedelta=lt//10) - forecasts_mean_eke.isel(prediction_timedelta=0)
    )

central_longitude = 180
lats, lons = eke_list[0].latitude.data, eke_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(12,3))

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 8, figure=fig)
axs_top, axs_bottom = [], []
eps = 0.0 # 1e-2

for i,lt in enumerate(lead_times_figure):
    im_top, ax = plot_map_on_grid(
        bias=eke_list[i], grid=eval(grid_list_top[i]), title='Ground Truth' if i == 0 else f't+{lt}', title_kwargs=dict(fontsize=11), 
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_top, norm=norm_top, levels=levels_top
    )
    axs_top.append(ax)

for i,lt in enumerate(lead_times_figure):
    if i == 0: 
        ax = fig.add_subplot(eval(grid_list_bottom[0]))
        ax.axis('off')
        continue
    im_bottom, ax = plot_map_on_grid(
        bias=eke_bias_list[i], grid=eval(grid_list_bottom[i]), title='', title_kwargs=dict(fontsize=11), # ='Ground Truth' if i == 0 else f't+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_bottom, norm=norm_bottom, levels=levels_bottom
    )
    axs_bottom.append(ax)

# single colorbar
cax_top = fig.add_axes([0.95, 0.55, 0.01, 0.4]) # [left, bottom, width, height]
cax_bottom = fig.add_axes([0.95, 0.05, 0.01, 0.4]) # [left, bottom, width, height]
cbar_top = fig.colorbar(
    im_top, ax=axs_top, cax=cax_top, orientation='vertical', label='m²/s²',
)
cbar_bottom = fig.colorbar(
    im_bottom, ax=axs_bottom, cax=cax_bottom, orientation='vertical', label='m²/s²',
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.0, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'eddy_kinetic_energy.pdf')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
lead_times_figure = [0, 10, 30, 60]
grid_list_top = [
    'gs[0,0:2]',
    'gs[0,2:4]',
    'gs[0,4:6]',
    'gs[0,6:8]',
]
grid_list_bottom = [
    'gs[1,0:2]',
    'gs[1,2:4]',
    'gs[1,4:6]',
    'gs[1,6:8]',
]

cmap_top, norm_top, levels_top = get_colormap_from_cmap('Spectral_r', 0.0, 0.1)
cmap_bottom, norm_bottom, levels_bottom = get_colormap_from_cmap('RdBu_r', -0.05, 0.05)

tke_list = []
tke_bias_list = []
for i,lt in enumerate(lead_times_figure):
    tke_list.append(
        forecasts_mean_tke.isel(prediction_timedelta=lt//10)
    )
    tke_bias_list.append(
        forecasts_mean_tke.isel(prediction_timedelta=lt//10) - forecasts_mean_tke.isel(prediction_timedelta=0)
    )

central_longitude = 180
lats, lons = tke_list[0].latitude.data, tke_list[0].longitude.data

fig, axes = plt.subplots(1, 1, figsize=(12,3))

axes.axis('off')

# initialize grid spec
gs = GridSpec(2, 8, figure=fig)
axs_top, axs_bottom = [], []
eps = 0.0 # 1e-2

for i,lt in enumerate(lead_times_figure):
    im_top, ax = plot_map_on_grid(
        bias=tke_list[i], grid=eval(grid_list_top[i]), title='Ground Truth' if i == 0 else f't+{lt}', title_kwargs=dict(fontsize=11), 
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_top, norm=norm_top, levels=levels_top
    )
    axs_top.append(ax)

for i,lt in enumerate(lead_times_figure):
    if i == 0: 
        ax = fig.add_subplot(eval(grid_list_bottom[0]))
        ax.axis('off')
        continue
    im_bottom, ax = plot_map_on_grid(
        bias=tke_bias_list[i], grid=eval(grid_list_bottom[i]), title='', title_kwargs=dict(fontsize=11), # ='Ground Truth' if i == 0 else f't+{lt}',
        fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_bottom, norm=norm_bottom, levels=levels_bottom
    )
    axs_bottom.append(ax)

# single colorbar
cax_top = fig.add_axes([0.95, 0.55, 0.01, 0.4]) # [left, bottom, width, height]
cax_bottom = fig.add_axes([0.95, 0.05, 0.01, 0.4]) # [left, bottom, width, height]
cbar_top = fig.colorbar(
    im_top, ax=axs_top, cax=cax_top, orientation='vertical', label='m²/s²',
)
cbar_bottom = fig.colorbar(
    im_bottom, ax=axs_bottom, cax=cax_bottom, orientation='vertical', label='m²/s²',
)
# adjust the plot
plt.subplots_adjust(
    left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.0, hspace=0.3
)

# save or plot figure
outfile = os.path.join(out_img_dir, f'total_kinetic_energy.pdf')
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
# lead_times_figure = [0, 10, 30, 60]
# grid_list_top = [
#     'gs[0:2,0]',
#     'gs[2:4,0]',
#     'gs[4:6,0]',
#     'gs[6:8,0]',
# ]
# grid_list_bottom = [
#     'gs[0:2,1]',
#     'gs[2:4,1]',
#     'gs[4:6,1]',
#     'gs[6:8,1]',
# ]

# cmap_top, norm_top, levels_top = get_colormap_from_cmap('Spectral_r', 0.0, 0.1)
# cmap_bottom, norm_bottom, levels_bottom = get_colormap_from_cmap('RdBu_r', -0.05, 0.05)

# eke_list = []
# eke_bias_list = []
# for i,lt in enumerate(lead_times_figure):
#     eke_list.append(
#         forecasts_mean_eke.isel(prediction_timedelta=lt//10)
#     )
#     eke_bias_list.append(
#         forecasts_mean_eke.isel(prediction_timedelta=lt//10) - forecasts_mean_eke.isel(prediction_timedelta=0)
#     )

# central_longitude = 180
# lats, lons = eke_list[0].latitude.data, eke_list[0].longitude.data

# fig, axes = plt.subplots(1, 1, figsize=(5,7))
# # fig.suptitle(f'Eddy Kinetic Energy', fontsize=16, y=1.05)

# axes.axis('off')

# # initialize grid spec
# gs = GridSpec(8, 2, figure=fig)
# axs_top, axs_bottom = [], []
# eps = 0.0 # 1e-2

# for i,lt in enumerate(lead_times_figure):
#     im_top, ax = plot_map_on_grid(
#         bias=eke_list[i], grid=eval(grid_list_top[i]), title='Ground Truth' if i == 0 else f't+{lt}', title_kwargs=dict(fontsize=11), 
#         fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_top, norm=norm_top, levels=levels_top
#     )
#     axs_top.append(ax)

# for i,lt in enumerate(lead_times_figure):
#     if i == 0: 
#         ax = fig.add_subplot(eval(grid_list_bottom[0]))
#         ax.axis('off')
#         continue
#     im_bottom, ax = plot_map_on_grid(
#         bias=eke_bias_list[i], grid=eval(grid_list_bottom[i]), title='', title_kwargs=dict(fontsize=11), # ='Ground Truth' if i == 0 else f't+{lt}',
#         fig=fig, lons=lons, lats=lats, central_longitude=central_longitude, cmap=cmap_bottom, norm=norm_bottom, levels=levels_bottom
#     )
#     axs_bottom.append(ax)

# # single colorbar
# cax_top = fig.add_axes([0.0, 0.0, 0.44, 0.025]) # [left, bottom, width, height]
# cax_bottom = fig.add_axes([0.5, 0.0, 0.44, 0.025]) # [left, bottom, width, height]
# cbar_top = fig.colorbar(
#     im_top, ax=axs_top, cax=cax_top, orientation='horizontal', label='m²/s²', ticks=[0.0, 0.025, 0.05, 0.075, 0.1],
# )
# cbar_bottom = fig.colorbar(
#     im_bottom, ax=axs_bottom, cax=cax_bottom, orientation='horizontal', label='m²/s²', ticks=[-0.05, -0.025, 0.0, 0.025, 0.05],
# )
# cbar_bottom.ax.tick_params() 
# # adjust the plot
# plt.subplots_adjust(
#     left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.3, hspace=0.05
# )

# # save or plot figure
# outfile = os.path.join(out_img_dir, f'eddy_kinetic_energy_vertical.pdf')
# plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
forecasts_mean_eke_timeseries = forecasts_eke.mean(dim=['latitude','longitude'], skipna=True).load()
forecasts_mean_tke_timeseries = forecasts_tke.mean(dim=['latitude','longitude'], skipna=True).load()

In [ ]:
colors = discrete_cmap(len(lead_times)+2, 'Blues_r')
colors = [colors(i) for i in range(len(lead_times))]

models_colors = {
    f'{lead_time}': colors[i]
    for i, lead_time in enumerate(lead_times)
}
models_names = {
    f'{lead_time}': 'Ground Truth' if lead_time == 0 else f"t + {lead_time}"
    for lead_time in lead_times
}
models_data = {}
for i,lt in enumerate(lead_times):
    models_data[f'{lt}'] = forecasts_mean_eke_timeseries.isel(prediction_timedelta=i)

outfile = os.path.join(out_img_dir, 'eddy_kinetic_energy_timeseries.png') if save_figures else None
plot_dmi_timeseries(
    data=models_data,
    colors=models_colors,
    names=models_names,
    dmi=None,
    threshold=None,
    dmi_label=False,
    outfile=outfile,
)

In [ ]:
colors = discrete_cmap(len(lead_times)+2, 'Blues_r')
colors = [colors(i) for i in range(len(lead_times))]

models_colors = {
    f'{lead_time}': colors[i]
    for i, lead_time in enumerate(lead_times)
}
models_names = {
    f'{lead_time}': 'Ground Truth' if lead_time == 0 else f"t + {lead_time}"
    for lead_time in lead_times
}
models_data = {}
for i,lt in enumerate(lead_times):
    models_data[f'{lt}'] = forecasts_mean_tke_timeseries.isel(prediction_timedelta=i)

outfile = os.path.join(out_img_dir, 'total_kinetic_energy_timeseries.png') if save_figures else None
plot_dmi_timeseries(
    data=models_data,
    colors=models_colors,
    names=models_names,
    dmi=None,
    threshold=None,
    dmi_label=False,
    outfile=outfile,
)

# Power Spectral Density of Eddy Kinetic Energy 

In [ ]:
from atmocean.utils import compute_psd_sht
import torch_harmonics as th
import torch

# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts['latitude'].shape[0], forecasts['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')
theta_min, Lmax, lambda_min

In [ ]:
psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_eke.isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

In [ ]:
outfile = os.path.join(out_img_dir, f'eddy_kinetic_energy_psd.png')

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# KUROSHIO
lat_slice, lon_slice = slice(50,20), slice(120,180)
# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice)['latitude'].shape[0], forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice)['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')

psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice).isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

outfile = os.path.join(out_img_dir, f'eke_power_spectral_density_kuroshio.png')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# AGULHAS
lat_slice, lon_slice = slice(-10,-50), slice(0,70)
# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice)['latitude'].shape[0], forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice)['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')

psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice).isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

outfile = os.path.join(out_img_dir, f'eke_power_spectral_density_agulhas.png')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# GULF STREAM
lat_slice, lon_slice = slice(60,20), slice(270,320)
# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice)['latitude'].shape[0], forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice)['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')

psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice).isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

outfile = os.path.join(out_img_dir, f'eke_power_spectral_density_gulf_stream.png')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# ANTARCTIC CIRCUMPOLAR CURRENT
lat_slice, lon_slice = slice(-40,-80), slice(0,360)
# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice)['latitude'].shape[0], forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice)['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')

psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice).isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

outfile = os.path.join(out_img_dir, f'eke_power_spectral_density_antarctic_circumpolar_current.png')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# BRAZIL-MALVINAS
lat_slice, lon_slice = slice(-20,-60), slice(280,340)
# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice)['latitude'].shape[0], forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice)['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')

psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_eke.sel(latitude=lat_slice, longitude=lon_slice).isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

outfile = os.path.join(out_img_dir, f'eke_power_spectral_density_brazil_malvinas_confluence.png')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

# Power Spectral Density of Total Kinetic Energy 

In [ ]:
from atmocean.utils import compute_psd_sht
import torch_harmonics as th
import torch

# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts['latitude'].shape[0], forecasts['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')
theta_min, Lmax, lambda_min

In [ ]:
psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_tke.isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

In [ ]:
outfile = os.path.join(out_img_dir, f'total_kinetic_energy_psd.png')

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# KUROSHIO
lat_slice, lon_slice = slice(50,20), slice(120,180)
# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice)['latitude'].shape[0], forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice)['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')

psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice).isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

outfile = os.path.join(out_img_dir, f'tke_power_spectral_density_kuroshio.png')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# AGULHAS
lat_slice, lon_slice = slice(-10,-50), slice(0,70)
# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice)['latitude'].shape[0], forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice)['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')

psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice).isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

outfile = os.path.join(out_img_dir, f'tke_power_spectral_density_agulhas.png')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# GULF STREAM
lat_slice, lon_slice = slice(60,20), slice(270,320)
# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice)['latitude'].shape[0], forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice)['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')

psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice).isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

outfile = os.path.join(out_img_dir, f'tke_power_spectral_density_gulf_stream.png')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# ANTARCTIC CIRCUMPOLAR CURRENT
lat_slice, lon_slice = slice(-40,-80), slice(0,360)
# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice)['latitude'].shape[0], forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice)['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')

psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice).isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

outfile = os.path.join(out_img_dir, f'tke_power_spectral_density_antarctic_circumpolar_current.png')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# BRAZIL-MALVINAS
lat_slice, lon_slice = slice(-20,-60), slice(280,340)
# earth radius
R_earth = 6371.0 # km
# number of lat-lon points (grid)
nlat, nlon = forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice)['latitude'].shape[0], forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice)['longitude'].shape[0]
# minimum resolving degrees
theta_min = 1.0
# cut frequency
Lmax = int(181 / theta_min)
# minimum spatial wavelength
lambda_min = (2 * np.pi * R_earth) / Lmax
# Spherical Harmonic Transform #
sht = th.RealSHT(nlat, nlon, lmax=Lmax, grid='equiangular')

psd_data_by_lead_time = {
    f'{lt}': compute_psd_sht(
        torch.as_tensor(np.nan_to_num(forecasts_mean_tke.sel(latitude=lat_slice, longitude=lon_slice).isel(prediction_timedelta=lt//10).values, nan=0)), sht, Lmax
    )
    for lt in lead_times
}

fig,ax = plt.subplots(1,1,figsize=(7,4))
for i,lt in enumerate(lead_times):
    ax.loglog(*psd_data_by_lead_time[f'{lt}'], label='Ground Truth' if i == 0 else f't+{lt}', color=colors[i])
ax.invert_xaxis()
plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
legend = ax.legend(loc='upper center', markerscale=20, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, -0.2))
for line in legend.get_lines():
    line.set_linewidth(4)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
ax.set_xlabel(f'Wavelength [km]', fontdict=dict(size=12))
ax.set_ylabel(f'Power Spectral Density', fontdict=dict(size=12))

outfile = os.path.join(out_img_dir, f'tke_power_spectral_density_brazil_malvinas_confluence.png')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()